In [1]:
import pandas as pd
import torch 
import torch.nn as nn
import torch.optim as optim

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.drop_duplicates(inplace = True)
df.shape
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
#Pre-processing

1.convert to Lowercase

In [5]:
df["review"] = df["review"].str.lower()

2.Removing the URLs

In [6]:
import re

In [7]:
def remove_urls(text):
    text = re.sub(r"https\S+", "",text) #(pattern,repl,string) eg:https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

In [8]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "",text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuation)

In [9]:
def remove_html(text):
    text = re.sub("<.*?>", "",text)
    return text

df["review"] = df["review"].apply(remove_html)

Removing Stopwords

In [10]:
import nltk #remove stopwaords,tokenization

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [12]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

6.Stemming

In [13]:
#porterStemming

from nltk.stem import PorterStemmer

In [14]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

7.Encoding

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [16]:
y = df["sentiment"]

In [17]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

8.Vectorization

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [26]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state = 42
)


In [27]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [30]:
from torch.utils.data import TensorDataset,DataLoader

In [31]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [32]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
test_loader = DataLoader(test_set,batch_size=64,shuffle=True)

In [47]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        #fully connected layer

        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self,x):
        #optional => shape(num of layers,batch size, hidden size)
        h0 = torch.zeros(self.num_layers,x.size(0), self.hidden_size)

        out, _ = self.rnn(x,h0)

        #1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        #2nd value = final hidden state of 

        out = self.fc(out[:,-1,:])
        return out

In [48]:
input_size = X_train.shape[1]

model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

Training the RNN

In [49]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) #add singleton direction

        outputs = model(Xb) #(batch_size,1) 

        outputs = torch.sigmoid(outputs.squeeze()) #batch_size => probability 

        loss = criterion(outputs, yb) #compute loss 
        loss.backward() #backpop
        optimizer.step() #weights and params are updated

    print(f"{epoch+1}/{epochs} and loss = {loss.item()}")

1/10 and loss = 0.3032597005367279
2/10 and loss = 0.27532094717025757
3/10 and loss = 0.32583707571029663
4/10 and loss = 0.14833715558052063
5/10 and loss = 0.1936839371919632
6/10 and loss = 0.36868637800216675
7/10 and loss = 0.20143504440784454
8/10 and loss = 0.17950153350830078
9/10 and loss = 0.13290007412433624
10/10 and loss = 0.21248877048492432


In [51]:
#eval 

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb,yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals = yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 13924.590163934425
